# Q3. Feature Engineering and Regression Pipeline

**Objective:** Build a reproducible scikit-learn regression pipeline to predict `items_sold` at a retail store.

**Dataset:** `q3_retail_promotions.csv`

This notebook covers:
- Date feature engineering
- Temporal train-test split
- Preprocessing using `ColumnTransformer`
- Linear Regression and Random Forest Regressor pipelines
- RMSE and MAE evaluation
- Parity plots
- Random Forest feature importances

## 1. Date Feature Engineering

In [1]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# Load the dataset
df = pd.read_csv("q3_retail_promotions.csv")

print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'q3_retail_promotions.csv'

In [ ]:
# Convert transaction_date to datetime
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

# Extract required date features
df["year"] = df["transaction_date"].dt.year
df["month"] = df["transaction_date"].dt.month
df["day_of_week"] = df["transaction_date"].dt.dayofweek

# Create binary month-end feature
df["is_month_end"] = np.where(df["transaction_date"].dt.day >= 25, 1, 0)

# Display sample to confirm new columns
df[[
    "transaction_date",
    "year",
    "month",
    "day_of_week",
    "is_month_end",
    "items_sold"
]].head(10)

### Date Feature Engineering Summary

The `transaction_date` column was converted into a datetime format and used to create four new features:

- `year`
- `month`
- `day_of_week`
- `is_month_end`

These features help the model capture time-based sales patterns such as monthly trends, weekday/weekend effects, and possible month-end purchase behaviour.

## 2. Temporal Train-Test Split

In [ ]:
# Sort data by transaction_date before splitting
df = df.sort_values("transaction_date").reset_index(drop=True)

# Use most recent 20% records as test set
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

print("\nTraining date range:")
print(train_df["transaction_date"].min(), "to", train_df["transaction_date"].max())

print("\nTest date range:")
print(test_df["transaction_date"].min(), "to", test_df["transaction_date"].max())

### Why Random Split is Inappropriate for Time-Ordered Data

A random train-test split is inappropriate for time-ordered data because it can mix future records into the training set. This creates data leakage, because the model may indirectly learn patterns from future observations that would not be available in a real forecasting situation.

Using the most recent 20% of records as the test set better simulates real-world prediction, where a model is trained on past data and evaluated on future unseen data.

## 3. Preprocessing Pipeline

In [ ]:
# Define target and features
target = "items_sold"

# Drop transaction_date because date information has already been engineered
feature_cols = [
    "store_id",
    "store_size",
    "location_type",
    "promotion_type",
    "is_weekend",
    "is_festival",
    "competition_density",
    "year",
    "month",
    "day_of_week",
    "is_month_end"
]

X_train = train_df[feature_cols]
y_train = train_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

# Define categorical and numerical columns
categorical_features = ["promotion_type", "location_type", "store_size"]

numerical_features = [
    "store_id",
    "is_weekend",
    "is_festival",
    "competition_density",
    "year",
    "month",
    "day_of_week",
    "is_month_end"
]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

In [ ]:
# Build preprocessing pipeline using ColumnTransformer
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features)
    ]
)

preprocessor

### Preprocessing Summary

The preprocessing step uses `ColumnTransformer` so that categorical and numerical features are handled correctly:

- `promotion_type`, `location_type`, and `store_size` are one-hot encoded.
- All numerical features are scaled using `StandardScaler`.

The preprocessing is placed inside the model pipeline, which ensures that transformations are fit only on the training data and then applied to the test data.

## 4. Model Training and Evaluation

In [ ]:
# Create model pipelines
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Train models
linear_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print("Linear Regression and Random Forest models trained successfully.")

In [ ]:
# Evaluation function
def evaluate_model(model_name, pipeline, X_test, y_test):
    predictions = pipeline.predict(X_test)

    rmse = mean_squared_error(y_test, predictions, squared=False)
    mae = mean_absolute_error(y_test, predictions)

    print(f"{model_name} Performance")
    print("-" * 40)
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")

    return predictions, rmse, mae

linear_pred, linear_rmse, linear_mae = evaluate_model(
    "Linear Regression",
    linear_pipeline,
    X_test,
    y_test
)

print()

rf_pred, rf_rmse, rf_mae = evaluate_model(
    "Random Forest Regressor",
    rf_pipeline,
    X_test,
    y_test
)

results_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "RMSE": [linear_rmse, rf_rmse],
    "MAE": [linear_mae, rf_mae]
})

results_df

### Parity Plot — Linear Regression

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, linear_pred, alpha=0.7)

min_value = min(y_test.min(), linear_pred.min())
max_value = max(y_test.max(), linear_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title("Parity Plot: Linear Regression")
plt.xlabel("Actual items_sold")
plt.ylabel("Predicted items_sold")
plt.grid(True)
plt.show()

**Interpretation:**  
In a parity plot, points closer to the diagonal reference line indicate better predictions. For Linear Regression, the spread of points around the diagonal shows how well the linear model captures the relationship between features and `items_sold`.

### Parity Plot — Random Forest Regressor

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, alpha=0.7)

min_value = min(y_test.min(), rf_pred.min())
max_value = max(y_test.max(), rf_pred.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title("Parity Plot: Random Forest Regressor")
plt.xlabel("Actual items_sold")
plt.ylabel("Predicted items_sold")
plt.grid(True)
plt.show()

**Interpretation:**  
The Random Forest parity plot shows how close the model's predictions are to the actual values. If the points are more tightly grouped around the diagonal line than Linear Regression, the Random Forest model is performing better.

### Random Forest Feature Importances

In [ ]:
# Get feature names after preprocessing
onehot_feature_names = rf_pipeline.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .get_feature_names_out(categorical_features)

all_feature_names = list(onehot_feature_names) + numerical_features

# Extract feature importances from Random Forest
rf_model = rf_pipeline.named_steps["model"]
feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Random Forest Feature Importances:")
feature_importance_df

In [ ]:
# Top 5 most influential features
top_5_features = feature_importance_df.head(5)

print("Top 5 Most Influential Features:")
top_5_features

In [ ]:
# Visualise top 5 feature importances
plt.figure(figsize=(8, 5))
plt.barh(top_5_features["feature"][::-1], top_5_features["importance"][::-1])
plt.title("Top 5 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Feature Importance Interpretation

The top 5 features shown above are the most influential predictors of `items_sold` according to the Random Forest model. These variables have the strongest contribution to reducing prediction error across the trees.

For example, if `promotion_type`, `is_festival`, or `competition_density` appear among the top features, it suggests that promotions, special events, and market competition strongly affect retail sales.

## Final Summary

This notebook completed the full regression pipeline workflow:

1. Loaded the retail promotions dataset.
2. Created date-based features from `transaction_date`.
3. Used a temporal train-test split instead of a random split.
4. Built preprocessing pipelines using `ColumnTransformer`.
5. Trained Linear Regression and Random Forest Regressor models.
6. Evaluated both models using RMSE and MAE.
7. Produced parity plots for both models.
8. Printed Random Forest feature importances and identified the top 5 influential features.